# Middleware

There are three different kinds of middleware:

- Agent Middleware: fires when an agent receives a run execution request (e.g,. executor-to-executor)
- Function Middleware: intercepts function calls within agents (e.g., tool calls)
- Chat Middleware: fires when a chat request is sent to AI models (e.g., within an agent)

Middleware only applies to agents: when they are called, when they call an LLM, and when they call a tool. There is no middleware for executors.

Middleware parameters are purely **framework-defined**, not user-defined. This means there is no point inspecting them.

In [1]:
import os
import inspect
import typing
import types
import pprint
from agent_framework import (
    AgentContext,
    FunctionInvocationContext,
    tool,
    Agent,
    MiddlewareTypes,
    AgentMiddleware,
    FunctionMiddleware,
    ChatMiddleware,
    ChatContext,
    agent_middleware
)
from agent_framework._middleware import categorize_middleware
from agent_framework._mcp import MCPTool
from agent_framework.openai import OpenAIChatClient
from typing import Never, Annotated, TypedDict, TYPE_CHECKING, cast, Any, TypeGuard, Literal, Callable, Awaitable
from pydantic import Field, BaseModel
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
class ToolParameterInfo(TypedDict):
    name: str
    type: list[type]
    required: bool
    default: object | None

class ToolInfo(TypedDict):
    name: str
    description: str
    approval_mode: str
    parameters: list[ToolParameterInfo]
    return_type: list[type]

MiddlewareCategory = Literal["agent", "function", "chat"]
ClassMiddlewareTypes = type[AgentMiddleware] | type[FunctionMiddleware] | type[ChatMiddleware]

class MiddlewareInfo(TypedDict):
    name: str
    description: str
    category: MiddlewareCategory


In [3]:
def _extract_types(tp) -> list[type]:
    if tp is None:
        return []
    
    origin = typing.get_origin(tp)
    if origin in (typing.Union, types.UnionType):
        return [t for t in typing.get_args(tp)]
    return [tp]

def get_functiontool_return_type(func: Callable) -> list[type]:
    hints = typing.get_type_hints(func)
    return _extract_types(hints.get("return", None))

def get_function_parameters(func: Callable) -> list[ToolParameterInfo]:
    sig = inspect.signature(func)
    hints = typing.get_type_hints(func)
    parameters_info = []
    
    for param_name, param in sig.parameters.items():
        param_type = hints.get(param_name, None)
        param_info = ToolParameterInfo(
            name=param_name,
            type=_extract_types(param_type),
            required=param.default is inspect.Parameter.empty,
            default=None if param.default is inspect.Parameter.empty else param.default
        )
        parameters_info.append(param_info)
    
    return parameters_info

def get_function_name(func: types.FunctionType) -> str:
    return func.__name__

def get_function_doc(func: types.FunctionType) -> str:
    if func.__doc__ is None:
        return ""
    return func.__doc__

def get_function_middleware_info(func: Callable, category: MiddlewareCategory) -> MiddlewareInfo:
    return MiddlewareInfo(
        name=get_function_name(func),
        description=get_function_doc(func),
        category=category,
    )

def get_class_middleware_info(cls: ClassMiddlewareTypes, category: MiddlewareCategory) -> MiddlewareInfo:
    return MiddlewareInfo(
        name=cls.__name__,
        description=inspect.getdoc(cls) or "",
        category=category,
    )

In [4]:
# Note: if you don't specify the name and description parameters in a @tool decorator, the framework will automatically use the function's name and docstring as fallbacks.
def get_previous_weather(
    location: Annotated[str, Field(description="The location to get the weather for.")], # required
    ctx: FunctionInvocationContext, # automatically detected by framework and marked as not required
    city: Annotated[str, Field(description="The city to get the weather for.")] = "Halifax", # not required
) -> str:
    """Get the previous weather for a given location."""
    return f"The previous weather in {location} was sunny with a high of 30°C."

@tool(name="weather_tool", description="Get the weather for a location", approval_mode="never_require")
def get_weather(
    location: Annotated[str, Field(description="The location to get the weather for.")], # required
    ctx: FunctionInvocationContext, # automatically detected by framework and marked as not required
    city: Annotated[str, Field(description="The city to get the weather for.")] = "Halifax", # not required
) -> str:
    """Get the weather for a given location."""
    return f"The weather in {location} is cloudy with a high of 15°C."

## Agent Middleware

Intercepts and modifies agent run execution (between agents/executors)


### Function-based

In [5]:
async def inject_tool_runtime_defaults(
    context: AgentContext,
    call_next: Callable[[], Awaitable[None]],
) -> None:
    """Agent middleware that sets tool-only runtime defaults."""
    print("[Agent] Starting execution")
    # context.function_invocation_kwargs.setdefault("tenant", "contoso")
    # context.function_invocation_kwargs.setdefault("request_source", "agent-middleware")

    await call_next()

    print("[Agent] Execution completed")

### Class-based

In [6]:
class InjectToolRuntimeDefaults(AgentMiddleware):
    """Agent middleware that sets tool-only runtime defaults."""

    # process is required and is the same signature and behaviour as function-based middleware
    async def process(
        self,
        context: AgentContext,
        call_next: Callable[[], Awaitable[None]],
    ) -> None:
        """Agent middleware that sets tool-only runtime defaults."""
        print("[Agent] Starting execution")
        # context.function_invocation_kwargs.setdefault("tenant", "contoso")
        # context.function_invocation_kwargs.setdefault("request_source", "agent-middleware")

        await call_next()

        print("[Agent] Execution completed")

## Function Middleware

Intercepts function (tool) calls within an agent

### Function-based

In [7]:
async def inject_function_kwargs(
    context: FunctionInvocationContext,
    call_next: Callable[[], Awaitable[None]],
) -> None:
    """Function middleware that injects function invocation kwargs."""
    # context.kwargs.setdefault("tenant", "contoso")
    # context.kwargs.setdefault("request_source", "function-middleware")

    await call_next()

### Class-based

In [8]:
class InjectFunctionKwargs(FunctionMiddleware):
    """Agent middleware that injects function invocation kwargs."""

    # process is required and is the same signature and behaviour as function-based middleware
    async def process(
        self,
        context: FunctionInvocationContext,
        call_next: Callable[[], Awaitable[None]],
    ) -> None:
        print("[Agent] Tool invocation starting")
        # context.function_invocation_kwargs.setdefault("tenant", "contoso")
        # context.function_invocation_kwargs.setdefault("request_source", "agent-middleware")

        await call_next()

        print("[Agent] Tool invocation completed")

## Chat Middleware

Chat middleware runs inside the function invocation loop. This means it executes for each model call, including calls that send tool results back to the model during a multi-turn tool calling sequence.

### Function-based

In [9]:
async def logging_chat_middleware(
    context: ChatContext,
    call_next: Callable[[], Awaitable[None]],
) -> None:
    """Chat middleware that logs AI interactions."""
    # Pre-processing: Log before AI call
    print(f"[Chat] Sending {len(context.messages)} messages to AI")

    # Continue to next middleware or AI service
    await call_next()

    # Post-processing: Log after AI response
    print("[Chat] AI response received")

### Class-based

In [10]:
class LoggingChatMiddleware(ChatMiddleware):
    """Chat middleware that logs AI interactions."""

    async def process(
        self,
        context: ChatContext,
        call_next: Callable[[], Awaitable[None]],
    ) -> None:
        print(f"[Chat Class] Sending {len(context.messages)} messages to AI")
        await call_next()
        print("[Chat Class] AI response received")

## Middleware Decorators

Convenience functions so you don't need to declare the types yourself

Note: only works for **function-based** middleware, not class-based.

In [11]:
@agent_middleware
async def simple_agent_middleware(context, call_next):
    """Simple agent middleware that logs before and after agent execution."""
    print("Before agent execution")
    await call_next()
    print("After agent execution")

In [12]:
chat_client = OpenAIChatClient(
    base_url=os.getenv("FOUNDRY_OPENAI_BASE_URL"),
    api_key=os.getenv("FOUNDRY_API_KEY"),
    model=os.getenv("FOUNDRY_MODEL"),
)

In [13]:
agent = Agent(
    client=chat_client,
    name="Tool agent",
    description="You are a tool agent that can utilize various tools to assist with tasks.",
    instructions="You have access to various tools and should use them to assist with tasks effectively.",
    tools=[get_weather, get_previous_weather],
    middleware=[
        inject_tool_runtime_defaults,
        InjectToolRuntimeDefaults(),
        inject_function_kwargs,
        InjectFunctionKwargs(),
        logging_chat_middleware,
        LoggingChatMiddleware(),
        simple_agent_middleware,
    ],
)

In [14]:
if agent.middleware is not None:
    CATEGORIES: tuple[MiddlewareCategory, ...] = ("agent", "function", "chat")
    categorized = categorize_middleware(agent.middleware)
    for category in CATEGORIES:
        for m in categorized[category]:
            if isinstance(m, (AgentMiddleware, FunctionMiddleware, ChatMiddleware)):
                print(get_class_middleware_info(type(m), category))
                # If needed, m.process holds the actual function to run.
                # print(get_function_parameters(m.process))
            else:
                print(get_function_middleware_info(m, category))

{'name': 'inject_tool_runtime_defaults', 'description': 'Agent middleware that sets tool-only runtime defaults.', 'category': 'agent'}
{'name': 'InjectToolRuntimeDefaults', 'description': 'Agent middleware that sets tool-only runtime defaults.', 'category': 'agent'}
{'name': 'simple_agent_middleware', 'description': 'Simple agent middleware that logs before and after agent execution.', 'category': 'agent'}
{'name': 'inject_function_kwargs', 'description': 'Function middleware that injects function invocation kwargs.', 'category': 'function'}
{'name': 'InjectFunctionKwargs', 'description': 'Agent middleware that injects function invocation kwargs.', 'category': 'function'}
{'name': 'logging_chat_middleware', 'description': 'Chat middleware that logs AI interactions.', 'category': 'chat'}
{'name': 'LoggingChatMiddleware', 'description': 'Chat middleware that logs AI interactions.', 'category': 'chat'}
